## UNIFICACIÓN DE DATOS DE WEB SCRAPING

In [14]:
# librerías
#!pip install demoji

import pandas as pd
import glob
import os
import re
#import demoji

In [15]:
# 1. Le decimos a Python dónde están los archivos 
ruta = "datos/scraping_hoteles/*.csv" 

# 2. Buscamos todos los archivos CSV de la carpeta
archivos_csv = glob.glob(ruta)
print(f"He encontrado {len(archivos_csv)} archivos CSV para unir.")

# 3. Creamos una lista vacía y vamos leyendo y guardando cada CSV
lista_dataframes = []

for archivo in archivos_csv:
    # Leemos cada archivo (suponemos que están separados por comas)
    df_temporal = pd.read_csv(archivo)
    lista_dataframes.append(df_temporal)

# 4. Unimos todos los recortes en una sola tabla gigante
df_total = pd.concat(lista_dataframes, ignore_index=True)
print(f"Total de reseñas combinadas en bruto: {len(df_total)}")

# 5. Eliminar duplicados (mismo hotel o reseña 2 veces)
df_total = df_total.drop_duplicates()

# 6. Ética y Privacidad (Borramos el nombre del usuario si existe)
# Cambia 'nombre_usuario' por el nombre exacto que le pusisteis a esa columna en Web Scraper
nombre_columna_usuario = 'nombre_usuario' 
if nombre_columna_usuario in df_total.columns:
    df_total = df_total.drop(columns=[nombre_columna_usuario])
    print("✓ Nombres de usuario borrados por privacidad (Cumplimiento RGPD).")


print(f"Total de reseñas finales listas para NLP: {len(df_total)}")

# 7. Exportamos el súper archivo final
df_total.to_csv("datos/procesados/df_hoteles_vlc_completo.csv", index=False, encoding='utf-8-sig')
print("¡Archivo 'df_hoteles_vlc_completo.csv' creado con éxito!")

He encontrado 8 archivos CSV para unir.
Total de reseñas combinadas en bruto: 58720
✓ Nombres de usuario borrados por privacidad (Cumplimiento RGPD).
Total de reseñas finales listas para NLP: 58720
¡Archivo 'df_hoteles_vlc_completo.csv' creado con éxito!


In [16]:
#hacemos una copia del dataframe original para no perderlo
df_original= df_total.copy()

## Preprocesado y limpieza del dataset

In [17]:
#eliminar columna de web order
df_total = df_total.drop(columns=['web_scraper_order'])
display(df_total.head())

,web_scraper_start_url,nombre_del_hotel,nacionalidad,detalles_hab,noches,personas,comentario_general,positivo,negativo,nota,fecha,nota_personal,nota_confort,nota_wifi,nota_instalaciones_servicios,nota_calidad_precio,nota_limpieza,nota_ubicacion,nota_media_resenas,comentario_media_hotel
0,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,Colombia,NaN,2 noches ·,En familia,Espectacular,Espectacular todo! Restaurante; servicio; habi...,Todo me gustó,10,Fecha del comentario: 14 de abril de 2026,"9,1","9,2","8,5","8,8","8,4","9,2","9,2","8,8",Fabuloso
1,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,1 noche ·,En pareja,Muy bien,.,.,"9,0",Fecha del comentario: 5 de abril de 2026,"9,1","9,2","8,5","8,8","8,4","9,2","9,2","8,8",Fabuloso
2,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,2 habitaciones Dobles Deluxe comunicadas,1 noche ·,En familia,Fantástico,"Habitaciones amplias, limpieza y ubicación",NaN,"9,0",Fecha del comentario: 3 de abril de 2026,"9,1","9,2","8,5","8,8","8,4","9,2","9,2","8,8",Fabuloso
3,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,1 noche ·,En pareja,Todo perfecto,El trato en recepción,El parking,10,Fecha del comentario: 21 de marzo de 2026,"9,1","9,2","8,5","8,8","8,4","9,2","9,2","8,8",Fabuloso
4,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,2 noches ·,En pareja,Una Nit del foc inolvidable.,Las vistas y el tamaño de la habitación.,"La calefacción central, sólo eso.",10,Fecha del comentario: 20 de marzo de 2026,"9,1","9,2","8,5","8,8","8,4","9,2","9,2","8,8",Fabuloso


In [18]:
# Vemos si hay NaNs en noches
print('NaNs en la columna noches antes del procesado:',df_total['noches'].isna().sum())
#de la columna noches extraemos el número de noches y actualizamos la columna, pasamos el digito a entero
df_total['noches'] = df_total['noches'].str.extract('(\d+)').astype(int)
# Comprobar si se ha creado algún NaN con el procesado
print('NaNs en la columna noches después del procesado:',df_total['noches'].isna().sum())

display(df_total.head())

NaNs en la columna noches antes del procesado: 0
NaNs en la columna noches después del procesado: 0


,web_scraper_start_url,nombre_del_hotel,nacionalidad,detalles_hab,noches,personas,comentario_general,positivo,negativo,nota,fecha,nota_personal,nota_confort,nota_wifi,nota_instalaciones_servicios,nota_calidad_precio,nota_limpieza,nota_ubicacion,nota_media_resenas,comentario_media_hotel
0,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,Colombia,NaN,2,En familia,Espectacular,Espectacular todo! Restaurante; servicio; habi...,Todo me gustó,10,Fecha del comentario: 14 de abril de 2026,"9,1","9,2","8,5","8,8","8,4","9,2","9,2","8,8",Fabuloso
1,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,1,En pareja,Muy bien,.,.,"9,0",Fecha del comentario: 5 de abril de 2026,"9,1","9,2","8,5","8,8","8,4","9,2","9,2","8,8",Fabuloso
2,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,2 habitaciones Dobles Deluxe comunicadas,1,En familia,Fantástico,"Habitaciones amplias, limpieza y ubicación",NaN,"9,0",Fecha del comentario: 3 de abril de 2026,"9,1","9,2","8,5","8,8","8,4","9,2","9,2","8,8",Fabuloso
3,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,1,En pareja,Todo perfecto,El trato en recepción,El parking,10,Fecha del comentario: 21 de marzo de 2026,"9,1","9,2","8,5","8,8","8,4","9,2","9,2","8,8",Fabuloso
4,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,2,En pareja,Una Nit del foc inolvidable.,Las vistas y el tamaño de la habitación.,"La calefacción central, sólo eso.",10,Fecha del comentario: 20 de marzo de 2026,"9,1","9,2","8,5","8,8","8,4","9,2","9,2","8,8",Fabuloso


In [19]:
#si la columna de negativo o positivo SOLO tiene un punto, lo convertimos a NA
df_total['negativo'] = df_total['negativo'].replace('.', pd.NA)
df_total['positivo'] = df_total['positivo'].replace('.', pd.NA)
display(df_total.head())

,web_scraper_start_url,nombre_del_hotel,nacionalidad,detalles_hab,noches,personas,comentario_general,positivo,negativo,nota,fecha,nota_personal,nota_confort,nota_wifi,nota_instalaciones_servicios,nota_calidad_precio,nota_limpieza,nota_ubicacion,nota_media_resenas,comentario_media_hotel
0,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,Colombia,NaN,2,En familia,Espectacular,Espectacular todo! Restaurante; servicio; habi...,Todo me gustó,10,Fecha del comentario: 14 de abril de 2026,"9,1","9,2","8,5","8,8","8,4","9,2","9,2","8,8",Fabuloso
1,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,1,En pareja,Muy bien,<NA>,<NA>,"9,0",Fecha del comentario: 5 de abril de 2026,"9,1","9,2","8,5","8,8","8,4","9,2","9,2","8,8",Fabuloso
2,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,2 habitaciones Dobles Deluxe comunicadas,1,En familia,Fantástico,"Habitaciones amplias, limpieza y ubicación",NaN,"9,0",Fecha del comentario: 3 de abril de 2026,"9,1","9,2","8,5","8,8","8,4","9,2","9,2","8,8",Fabuloso
3,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,1,En pareja,Todo perfecto,El trato en recepción,El parking,10,Fecha del comentario: 21 de marzo de 2026,"9,1","9,2","8,5","8,8","8,4","9,2","9,2","8,8",Fabuloso
4,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,2,En pareja,Una Nit del foc inolvidable.,Las vistas y el tamaño de la habitación.,"La calefacción central, sólo eso.",10,Fecha del comentario: 20 de marzo de 2026,"9,1","9,2","8,5","8,8","8,4","9,2","9,2","8,8",Fabuloso


In [20]:
#todas las columnas de nota cambiamos las comas por puntos y las convertimos a numéricas
#coger las columnas que empiezan por 'nota'
columnas_notas = [col for col in df_total.columns if col.startswith('nota')]
# Comprobamos posibles NaNs antes del preprocesado
print('NaNs en las columnas de notas antes del procesado:',df_total[columnas_notas].isna().sum().sum())
for columna in columnas_notas:
    df_total[columna] = df_total[columna].str.replace(',', '.')
    df_total[columna] = pd.to_numeric(df_total[columna], errors='coerce')
# Vemos si se han creado nuevos NaNs
print('NaNs en las columnas de notas después del procesado:',df_total[columnas_notas].isna().sum().sum())
display(df_total.head())

NaNs en las columnas de notas antes del procesado: 3354
NaNs en las columnas de notas después del procesado: 3354


,web_scraper_start_url,nombre_del_hotel,nacionalidad,detalles_hab,noches,personas,comentario_general,positivo,negativo,nota,fecha,nota_personal,nota_confort,nota_wifi,nota_instalaciones_servicios,nota_calidad_precio,nota_limpieza,nota_ubicacion,nota_media_resenas,comentario_media_hotel
0,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,Colombia,NaN,2,En familia,Espectacular,Espectacular todo! Restaurante; servicio; habi...,Todo me gustó,10.0,Fecha del comentario: 14 de abril de 2026,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8,Fabuloso
1,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,1,En pareja,Muy bien,<NA>,<NA>,9.0,Fecha del comentario: 5 de abril de 2026,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8,Fabuloso
2,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,2 habitaciones Dobles Deluxe comunicadas,1,En familia,Fantástico,"Habitaciones amplias, limpieza y ubicación",NaN,9.0,Fecha del comentario: 3 de abril de 2026,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8,Fabuloso
3,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,1,En pareja,Todo perfecto,El trato en recepción,El parking,10.0,Fecha del comentario: 21 de marzo de 2026,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8,Fabuloso
4,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,2,En pareja,Una Nit del foc inolvidable.,Las vistas y el tamaño de la habitación.,"La calefacción central, sólo eso.",10.0,Fecha del comentario: 20 de marzo de 2026,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8,Fabuloso


In [21]:
# Comprobar NaNs
print('NaNs en la columna fecha antes del procesado:',df_total['fecha'].isna().sum())
# Eliminar parte "Fecha del comentario: " y espacios al principio o final
df_total['fecha'] = df_total['fecha'].apply(lambda x: re.sub('Fecha del comentario: ', '', x).strip())
# Pasar los meses en español a un formato conocido para pandas
mes_a_codigo = {'enero': '01', 'febrero': '02', 'marzo': '03', 'abril': '04', 'mayo': '05', 'junio': '06', 'julio': '07', 'agosto': '08',
                'septiembre': '09', 'octubre': '10', 'noviembre': '11', 'diciembre': '12'}
for mes in mes_a_codigo.keys():
    df_total['fecha'] = df_total['fecha'].apply(lambda x: re.sub(mes, mes_a_codigo[mes], x))
# Pasar a formato fecha
df_total['fecha'] = pd.to_datetime(df_total['fecha'], format='%d de %m de %Y', errors='coerce')
# Comprobar NaNs
print('NaNs en la columna fecha después del procesado:',df_total['fecha'].isna().sum())
display(df_total.head())

NaNs en la columna fecha antes del procesado: 0
NaNs en la columna fecha después del procesado: 0


,web_scraper_start_url,nombre_del_hotel,nacionalidad,detalles_hab,noches,personas,comentario_general,positivo,negativo,nota,fecha,nota_personal,nota_confort,nota_wifi,nota_instalaciones_servicios,nota_calidad_precio,nota_limpieza,nota_ubicacion,nota_media_resenas,comentario_media_hotel
0,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,Colombia,NaN,2,En familia,Espectacular,Espectacular todo! Restaurante; servicio; habi...,Todo me gustó,10.0,2026-04-14,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8,Fabuloso
1,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,1,En pareja,Muy bien,<NA>,<NA>,9.0,2026-04-05,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8,Fabuloso
2,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,2 habitaciones Dobles Deluxe comunicadas,1,En familia,Fantástico,"Habitaciones amplias, limpieza y ubicación",NaN,9.0,2026-04-03,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8,Fabuloso
3,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,1,En pareja,Todo perfecto,El trato en recepción,El parking,10.0,2026-03-21,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8,Fabuloso
4,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,2,En pareja,Una Nit del foc inolvidable.,Las vistas y el tamaño de la habitación.,"La calefacción central, sólo eso.",10.0,2026-03-20,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8,Fabuloso


In [22]:
# Problemas al pasar la fecha así
# #cogemos la columna de fecha, queremos quedarnos SOLO con la fecha y despues convertirla a formato datetime
# # Comprobamos NaNs antes del preprocesado
# print('NaNs en la columna fecha antes del procesado:',df_total['fecha'].isna().sum())
# #ahora las celdas son estilo: "Fecha del comentario, 14 de marzo de 2023"
# df_total['fecha'] = df_total['fecha'].str.extract('(\d{1,2} de \w+ de \d{4})')
# df_total['fecha'] = pd.to_datetime(df_total['fecha'], format='%d de %B de %Y', errors='coerce')
# #quitar la hora de la fecha, solo queremos la fecha
# df_total['fecha'] = df_total['fecha'].dt.date
# # Comprobamos NaNs después del preprocesado
# print('NaNs en la columna fecha después del procesado:',df_total['fecha'].isna().sum())
# display(df_total.head())

In [23]:
#quitar el "En" de la columna de personas, ahora tiene formato "En familia", "En pareja", "Solo", etc. Queremos quedarnos solo con el tipo de persona, sin el "En"
df_total['personas'] = df_total['personas'].str.replace('En ', '')
display(df_total.head())

,web_scraper_start_url,nombre_del_hotel,nacionalidad,detalles_hab,noches,personas,comentario_general,positivo,negativo,nota,fecha,nota_personal,nota_confort,nota_wifi,nota_instalaciones_servicios,nota_calidad_precio,nota_limpieza,nota_ubicacion,nota_media_resenas,comentario_media_hotel
0,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,Colombia,NaN,2,familia,Espectacular,Espectacular todo! Restaurante; servicio; habi...,Todo me gustó,10.0,2026-04-14,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8,Fabuloso
1,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,1,pareja,Muy bien,<NA>,<NA>,9.0,2026-04-05,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8,Fabuloso
2,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,2 habitaciones Dobles Deluxe comunicadas,1,familia,Fantástico,"Habitaciones amplias, limpieza y ubicación",NaN,9.0,2026-04-03,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8,Fabuloso
3,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,1,pareja,Todo perfecto,El trato en recepción,El parking,10.0,2026-03-21,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8,Fabuloso
4,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,España,NaN,2,pareja,Una Nit del foc inolvidable.,Las vistas y el tamaño de la habitación.,"La calefacción central, sólo eso.",10.0,2026-03-20,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8,Fabuloso


Vamos a crear un csv aparte que contenga la información del hotel y otro csv con la información de los comentarios, para luego unirlos por el id del hotel.
Para el csv de los hoteles, vamos a quedarnos con las columnas: web_scraper_start_url, nombre_del_hotel, y las que empiecen por "nota_". Para el csv de los comentarios, nos quedaremos con el resto de columnas, incluyendo el id del hotel que identificará cada hotel en el primer csv. 

In [24]:
#Vamos a crear un csv aparte que contenga la información del hotel y otro csv con la información de los comentarios, para luego unirlos por el id del hotel.
#Para el csv de los hoteles, vamos a quedarnos con las columnas: web_scraper_start_url, nombre_del_hotel, y las que empiecen por "nota_". Para el csv de los comentarios, nos quedaremos con el resto de columnas, incluyendo el id del hotel que identificará cada hotel en el primer csv. 
#el id del hotel viene del nombre del hotel, que es único para cada hotel, así que lo usaremos como id para unir los csv después.

#creamos el csv de hoteles
columnas_hoteles = ['web_scraper_start_url', 'nombre_del_hotel'] + [col for col in df_total.columns if col.startswith('nota_')]
df_hoteles = df_total[columnas_hoteles].drop_duplicates()
df_hoteles.to_csv("datos/procesados/df_hoteles_vlc_info.csv", index=False, encoding='utf-8-sig')
print("¡Archivo 'df_hoteles_vlc_info.csv' creado con éxito!")

#creamos el csv de comentarios, dejar la columna de nombre del hotel para poder unir luego
columnas_comentarios = ['nombre_del_hotel'] + [col for col in df_total.columns if col not in columnas_hoteles]
df_comentarios = df_total[columnas_comentarios]
df_comentarios.to_csv("datos/procesados/df_hoteles_vlc_comentarios.csv", index=False, encoding='utf-8-sig')
print("¡Archivo 'df_hoteles_vlc_comentarios.csv' creado con éxito!")

¡Archivo 'df_hoteles_vlc_info.csv' creado con éxito!
¡Archivo 'df_hoteles_vlc_comentarios.csv' creado con éxito!


In [25]:
display(df_comentarios)

,nombre_del_hotel,nacionalidad,detalles_hab,noches,personas,comentario_general,positivo,negativo,nota,fecha,comentario_media_hotel
0,Barceló Valencia,Colombia,NaN,2,familia,Espectacular,Espectacular todo! Restaurante; servicio; habi...,Todo me gustó,10.0,2026-04-14,Fabuloso
1,Barceló Valencia,España,NaN,1,pareja,Muy bien,<NA>,<NA>,9.0,2026-04-05,Fabuloso
2,Barceló Valencia,España,2 habitaciones Dobles Deluxe comunicadas,1,familia,Fantástico,"Habitaciones amplias, limpieza y ubicación",NaN,9.0,2026-04-03,Fabuloso
3,Barceló Valencia,España,NaN,1,pareja,Todo perfecto,El trato en recepción,El parking,10.0,2026-03-21,Fabuloso
4,Barceló Valencia,España,NaN,2,pareja,Una Nit del foc inolvidable.,Las vistas y el tamaño de la habitación.,"La calefacción central, sólo eso.",10.0,2026-03-20,Fabuloso
...,...,...,...,...,...,...,...,...,...,...,...
58715,NH Valencia Las Ciencias,Alemania,NaN,1,pareja,Muy bien,NaN,NaN,8.0,2023-05-29,Bien
58716,NH Valencia Las Ciencias,Italia,Habitación Doble Estándar con cama supletoria ...,2,familia,Excepcional,NaN,NaN,10.0,2023-11-09,Bien
58717,NH Valencia Las Ciencias,España,Habitación Doble Estándar con cama supletoria ...,3,familia,Excepcional,NaN,NaN,10.0,2023-10-20,Bien
58718,NH Valencia Las Ciencias,Rumanía,NaN,3,pareja,Perfect,NaN,NaN,10.0,2023-07-09,Bien


In [26]:
display(df_hoteles)

,web_scraper_start_url,nombre_del_hotel,nota_personal,nota_confort,nota_wifi,nota_instalaciones_servicios,nota_calidad_precio,nota_limpieza,nota_ubicacion,nota_media_resenas
0,https://www.booking.com/hotel/es/barcelo-valen...,Barceló Valencia,9.1,9.2,8.5,8.8,8.4,9.2,9.2,8.8
2108,https://www.booking.com/hotel/es/venecia.es.ht...,Venecia Plaza Centro,9.5,9.4,8.9,9.1,9.0,9.4,9.9,9.2
11985,https://www.booking.com/hotel/es/las-arenas-ca...,Hotel Las Arenas Balneario Resort,8.9,8.9,8.3,8.6,8.5,8.9,9.2,8.6
14985,https://www.booking.com/hotel/es/casual-valenc...,Casual Vintage Valencia,9.3,8.6,8.6,8.2,8.4,8.7,9.8,8.4
22302,https://www.booking.com/hotel/es/only-you-vale...,Only YOU Hotel Valencia,9.7,9.6,9.1,9.4,8.9,9.6,9.8,9.4
27639,https://www.booking.com/hotel/es/casual-valenc...,Casual del Cine Valencia by Casual Hoteles,9.4,8.7,8.8,8.3,8.4,8.8,9.7,8.5
31601,https://www.booking.com/hotel/es/melia-valenci...,Meliá Valencia,9.0,9.1,NaN,8.8,8.3,9.0,8.3,8.6
34955,https://www.booking.com/hotel/es/the-river-hos...,The River Hostel,8.9,8.3,8.2,8.3,8.6,8.4,9.1,8.2
41489,https://www.booking.com/hotel/es/renasa.es.htm...,Sweet Hotel Renasa,8.8,8.4,8.0,8.0,7.9,8.5,8.3,8.1
45480,https://www.booking.com/hotel/es/puertavalenci...,Silken Puerta Valencia,9.0,8.9,8.2,8.6,8.3,9.0,8.4,8.6


## Limpieza (minúsculas, enlaces y caracteres, emoticonos)

In [27]:
# Traducir emojis
!pip install emoji --upgrade
# Los traduce al inglés (tendremos que mirar si los traducimos al español todos o a qué idioma)
""" def handle_emoji(string):
    if not isinstance(string, str):
        return string
    emojis = demoji.findall(string)
    for emoji in emojis:
        string = string.replace(emoji, " [" + emojis[emoji].split(":")[0].strip() + "] ")
    return string
 """
import emoji

def handle_emoji(string):
    if not isinstance(string, str):
        return string
        
    # Buscamos los emojis únicos presentes en la cadena
    emojis_encontrados = set(c['emoji'] for c in emoji.emoji_list(string))
    
    for emj in emojis_encontrados:
        # demojize lo convierte a español con formato ":cara_sonriendo:"
        nombre_es = emoji.demojize(emj, language='es')
        
        # Limpiamos el texto: quitamos los dos puntos y cambiamos guiones bajos por espacios
        texto_limpio = nombre_es.replace(':', '').replace('_', ' ')
        
        # EL CAMBIO ESTÁ AQUÍ: Reemplazamos el emoji sin usar corchetes, solo con espacios
        string = string.replace(emj, f" {texto_limpio} ")
        
    # Opcional: Esto elimina los espacios dobles que puedan quedar si el emoji ya estaba rodeado de espacios
    string = " ".join(string.split())
        
    return string

# Ejemplo de uso:
texto = "Me encanta Python 😍 y tomar café ☕"
print(handle_emoji(texto))

string_cols = df_comentarios.select_dtypes(include=['object', 'string']).columns
df_comentarios[string_cols] = df_comentarios[string_cols].apply(lambda col: col.apply(handle_emoji))

# Todo a minúscula
string_cols = df_comentarios.select_dtypes(include=['object', 'string']).columns
df_comentarios[string_cols] = df_comentarios[string_cols].apply(lambda col: col.str.lower())

# Eliminar enlaces y limpiar espacios
string_cols = df_comentarios.select_dtypes(include=['object', 'string']).columns
df_comentarios[string_cols] = df_comentarios[string_cols].apply(lambda col: col.str.replace(r'https?://\S+|www\.\S+', '', regex=True).str.strip())

# Eliminar caracteres especiales
df_comentarios[string_cols] = df_comentarios[string_cols].apply(lambda col: col.str.replace(r'[^a-záéíóúüñàèìòùâêîôûäëïöüa-z0-9\s\[\].,!?;:\-]', ' ', regex=True))

Me encanta Python cara sonriendo con ojos de corazón y tomar café bebida caliente


C:\Users\adrim\AppData\Local\Temp\ipykernel_11008\1141338288.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_comentarios[string_cols] = df_comentarios[string_cols].apply(lambda col: col.apply(handle_emoji))
C:\Users\adrim\AppData\Local\Temp\ipykernel_11008\1141338288.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_comentarios[string_cols] = df_comentarios[string_cols].apply(lambda col: col.str.lower())
C:\Users\adrim\AppData\Local\Temp\ipykernel_11008\1141338288.py:50: SettingWithCopyWarning

In [28]:
display(df_comentarios)

,nombre_del_hotel,nacionalidad,detalles_hab,noches,personas,comentario_general,positivo,negativo,nota,fecha,comentario_media_hotel
0,barceló valencia,colombia,NaN,2,familia,espectacular,espectacular todo! restaurante; servicio; habi...,todo me gustó,10.0,2026-04-14,fabuloso
1,barceló valencia,españa,NaN,1,pareja,muy bien,<NA>,<NA>,9.0,2026-04-05,fabuloso
2,barceló valencia,españa,2 habitaciones dobles deluxe comunicadas,1,familia,fantástico,"habitaciones amplias, limpieza y ubicación",NaN,9.0,2026-04-03,fabuloso
3,barceló valencia,españa,NaN,1,pareja,todo perfecto,el trato en recepción,el parking,10.0,2026-03-21,fabuloso
4,barceló valencia,españa,NaN,2,pareja,una nit del foc inolvidable.,las vistas y el tamaño de la habitación.,"la calefacción central, sólo eso.",10.0,2026-03-20,fabuloso
...,...,...,...,...,...,...,...,...,...,...,...
58715,nh valencia las ciencias,alemania,NaN,1,pareja,muy bien,NaN,NaN,8.0,2023-05-29,bien
58716,nh valencia las ciencias,italia,habitación doble estándar con cama supletoria ...,2,familia,excepcional,NaN,NaN,10.0,2023-11-09,bien
58717,nh valencia las ciencias,españa,habitación doble estándar con cama supletoria ...,3,familia,excepcional,NaN,NaN,10.0,2023-10-20,bien
58718,nh valencia las ciencias,rumanía,NaN,3,pareja,perfect,NaN,NaN,10.0,2023-07-09,bien


In [29]:
df_comentarios.to_csv("datos/procesados/df_comentarios_limpio.csv", index=False, encoding='utf-8-sig')
print("¡Archivo 'df_comentarios_limpio.csv' creado con éxito!")

¡Archivo 'df_comentarios_limpio.csv' creado con éxito!
